# DTW Analysis

In [2]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [3]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from Analysis.DTW_Incidents.dtw_functions import (
    annotate_pylon_unmatched_reasons,
    combine_incidents,
    merge_dtw_daily_with_pylon,
)
from general_functions.call_api_with_account_id import call_api_with_accountId
from general_functions.return_workspace_ids import return_workspace_ids

logger = logging.getLogger(__name__)


In [4]:
url = "https://targeting.innkeepr.ai/api/"
workspace_ids = {ws["name"]: ws["id"] for ws in return_workspace_ids()}
pylon_path = "Analysis/DTW_Incidents/2026-07-20_pylon_tickets_dtw_only.csv"
pylon_states = ['closed', 'new', 'waiting_on_you', 'on_hold']

# Load data

In [5]:
dtw_daily = pd.read_csv("Analysis/DTW_Incidents/2026-07-20-dtw_incident_daily-3.csv")
dtw_daily["date"] = pd.to_datetime(dtw_daily["date"])
dtw_daily["is_incident"] = dtw_daily["is_incident"].astype(bool)
dtw_daily["week"] = dtw_daily["date"].dt.isocalendar().week.astype(int)
dtw_daily["year"] = dtw_daily["date"].dt.isocalendar().year.astype(int)
dtw_daily["month"] = dtw_daily["date"].dt.month.astype(int)
print(f"Date range: {dtw_daily['date'].min()} to {dtw_daily['date'].max()}")
dtw_daily.head()

In [6]:
number_workspaces = dtw_daily["account_name"].unique()
print(f"Number of workspaces: {len(number_workspaces)}")
sorted(number_workspaces)

# Preprocess Data

In [7]:
# check unknown model types
unknown_model_type = dtw_daily[dtw_daily["model_type"]=="unknown"]
unknown_signals = unknown_model_type[["account_name","audience_id","date"]].drop_duplicates().to_dict(orient="records")
missing_models = pd.DataFrame(columns=["account_name","audience_id","model_created","model_type"])
for signal in unknown_signals:
    workspace = signal["account_name"]
    audience_id = signal["audience_id"]
    date = signal["date"]
    print(f"Find model for {workspace} on {date} and audience {audience_id}")
    workspace_id = workspace_ids[workspace]
    signal_def = call_api_with_accountId(f"{url}signals/query", workspace_id, {"id":audience_id}, logger)
    signal_def = signal_def[0]
    # if signal is archived or has no model, drop from data
    if signal_def.get("status", None) in [None, "archived"] and signal_def.get("model", None) is None:
        print(f"... This signal is archived or has no model. Skipping. Drop from data")
        dtw_daily = dtw_daily[dtw_daily["audience_id"]!=audience_id]
        continue
    raise ValueError("Stop here - Adding models not implemented here")
unknown_models = dtw_daily[dtw_daily["model_type"]=="unknown_signals"]
if unknown_models.shape[0]>0:
    raise ValueError("Stop here - Models are missing")

In [8]:
# add workspace id
dtw_daily["workspace_id"] = dtw_daily["account_name"].map(workspace_ids)

In [9]:
# add source to data
def return_connection(audience_id, workspace_id):
    url_signals = f"{url}signals/query"
    content = {"id":audience_id}
    signal_def = call_api_with_accountId(url_signals, workspace_id, content, logger)
    connection =  signal_def[0].get("connection", {}).get("platform", {}).get("name",None)
    if connection is None:
        raise ValueError("Stop here - Connection is missing")
    return connection
workspace_signals = dtw_daily[["workspace_id","audience_id"]].drop_duplicates()
print(f"workspace_signals rows: {len(workspace_signals)}")
workspace_signals["connection"] = workspace_signals.apply(lambda x: return_connection(x["audience_id"], x["workspace_id"]), axis=1)

dtw_daily = pd.merge(dtw_daily, workspace_signals, on=["workspace_id","audience_id"], how="left")

In [10]:
dtw_daily["connection"].value_counts(dropna=False)

## Merge DTW daily with Pylon tickets

Match on `audience_id` ↔ `signal_id`, `workspace_id`, and `dtw_daily.date` ↔ `pylon.created_at` (date).


In [11]:
pylon = pd.read_csv(pylon_path)

dtw_daily_pylon, pylon_unmatched = merge_dtw_daily_with_pylon(
    dtw_daily,
    pylon,
)

print(f"dtw_daily rows: {len(dtw_daily)}")
print(f"pylon tickets: {len(pylon)}")
print(f"dtw rows with pylon match: {dtw_daily_pylon['issue_id'].notna().sum()}")
print(f"pylon tickets matched: {len(pylon) - len(pylon_unmatched)}")
print(f"pylon tickets unmatched: {len(pylon_unmatched)}")

dtw_daily_pylon.head()


### Pylon Tickets which could not be merged to an incident

In [12]:
pylon_unmatched = annotate_pylon_unmatched_reasons(
    pylon_unmatched,
    dtw_daily_pylon[["audience_id", "workspace_id", "date"]].drop_duplicates(),
)

print("Unmatched by reason:")
print(pylon_unmatched["unmatched_reason"].value_counts(dropna=False).to_string())
print()
pylon_unmatched[
    [
        "issue_id",
        "number",
        "title",
        "state",
        "created_at",
        "created_date",
        "signal_id",
        "workspace_id",
        "incident_date",
        "unmatched_reason",
        "issue_url",
    ]
]


In [13]:
# ticket was created by me: handle case somehow dtw_daily[(dtw_daily["audience_id"]=="68dd4512120f694f539c0faa")]#(dtw_daily["workspace_id"]=="68b9534c5287866a2c94a8f5")
# dtw_daily[(dtw_daily["audience_id"]=="6a1fca961370504ba6a56eef")]#&(dtw_daily["workspace_id"]=="6870b934768354324d58e9cf")]


## Handle Incidents with Pylon Tickets

In [14]:
dtw_daily_pylon.to_csv("Analysis/DTW_Incidents/test_data.csv")

# Analysis

## Conversion vs. Causal Models

In [15]:
model_type_incidents = dtw_daily_pylon.groupby(by=["model_type", "connection","is_incident"]).agg(
    count=("is_incident", "count"),
).reset_index()
model_type_incidents = model_type_incidents.pivot(index=["model_type", "connection"], columns="is_incident", values="count")
model_type_incidents["ratio"] = model_type_incidents[True] / (model_type_incidents[False]+model_type_incidents[True])
model_type_incidents.reset_index(inplace=True)#.drop(columns=["is_incident"])
model_type_incidents

## Weekly Incidents

In [16]:
weekly_incidents_workspaces = (
    dtw_daily_pylon.groupby(
        by=["account_name", "year", "week", "model_type"],
        as_index=False,
    )
    .agg(
        incident_days=("is_incident", "sum"),
        total_days=("is_incident", "count"),
    )
)
weekly_incidents_workspaces["incident_days"] = weekly_incidents_workspaces["incident_days"].astype(int)
weekly_incidents_workspaces["non_incident_days"] = (
    weekly_incidents_workspaces["total_days"] - weekly_incidents_workspaces["incident_days"]
)
weekly_incidents_workspaces["is_incident"] = (
    weekly_incidents_workspaces["incident_days"] / weekly_incidents_workspaces["total_days"]
)

weekly_incidents_workspaces["year-week"] = (
    weekly_incidents_workspaces["year"].astype(str)
    + "-"
    + weekly_incidents_workspaces["week"].astype(str).str.zfill(2)
)
weekly_incidents_workspaces = weekly_incidents_workspaces.sort_values(
    by=["year", "week"]
).reset_index(drop=True)

# create pivot table
weekly_incidents_workspaces_pivot = weekly_incidents_workspaces.pivot(index=["year-week","model_type"], columns="account_name", values="is_incident")
weekly_incidents_workspaces_pivot=weekly_incidents_workspaces_pivot.reset_index().sort_values(by=["model_type","year-week"])
weekly_incidents_workspaces_pivot

### Overall weekly Incident Rates

In [17]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
plt.suptitle("Overall Incident Rates")
ax1.set_title("Incident Rate by Model Type")
sns.barplot(data=weekly_incidents_workspaces, x="year-week", y="is_incident", hue="model_type", ax=ax1, errorbar="sd")
plt.grid(True)
ax2.set_title("Incident Days by Model Type")
sns.barplot(data=weekly_incidents_workspaces, x="year-week", y="incident_days", hue="model_type", ax=ax2, errorbar="sd")
plt.grid(True)


In [18]:
sns.set_theme(style="whitegrid", context="notebook")

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=weekly_incidents_workspaces,
    x="account_name",
    y="is_incident",
    hue="model_type",
    ax=ax,
)
ax.set_title("Weekly incident rate by account and model type")
ax.set_xlabel("Account")
ax.set_ylabel("Incident rate")
ax.set_ylim(-0.05, 1.05)
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Model type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()

### Weekly Pylon Tags

In [19]:
dtw_daily_pylon["tags"] = np.where(
    (dtw_daily_pylon["tags"].isna())&(dtw_daily_pylon["issue_id"].notna()),
    "no_tags",
    dtw_daily_pylon["tags"]
)
dtw_daily_pylon["tags"] = np.where(
    (dtw_daily_pylon["tags"].isna())&(dtw_daily_pylon["issue_id"].isna()),
    "no_pylon_ticket",
    dtw_daily_pylon["tags"]
)
print(dtw_daily_pylon["tags"].value_counts(dropna=False))
weekly_tags = dtw_daily_pylon.groupby(by=["year", "week", "model_type","tags","connection"]).agg(
    incident_days=("is_incident", "sum"),
).reset_index()
weekly_tags["total_days"] = weekly_tags.groupby(by=["year", "week", "model_type"])["incident_days"].transform("sum")
weekly_tags["is_incident"] = weekly_tags["incident_days"] / weekly_tags["total_days"]
weekly_tags["year-week"] = weekly_tags["year"].astype(str) + "-" + weekly_tags["week"].astype(str).str.zfill(2)
weekly_tags


In [20]:
vc_tags = dtw_daily_pylon[dtw_daily_pylon["tags"]!="no_pylon_ticket"]["tags"].value_counts(dropna=False)
vc_tags.plot(kind="bar")
plt.title("Pylon tags")

# Applied Change by Connection & Count

In [21]:
plot_df_conversion = weekly_tags[
    (weekly_tags["tags"] != "no_pylon_ticket")
    & (weekly_tags["model_type"] == "conversion")
].copy()
plot_df_conversion["incident_days"] = plot_df_conversion["incident_days"].astype(int)

weeks = sorted(plot_df_conversion["year-week"].unique())
tags = sorted(plot_df_conversion["tags"].unique())
week_to_x = {w: i for i, w in enumerate(weeks)}
tag_to_y = {t: i for i, t in enumerate(tags)}

min_c, max_c = plot_df_conversion["incident_days"].min(), plot_df_conversion["incident_days"].max()

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xticks(range(len(weeks)))
ax.set_xticklabels(weeks, rotation=90)
ax.set_yticks(range(len(tags)))
ax.set_yticklabels(tags)
ax.set_xlim(-0.5, len(weeks) - 0.5)
ax.set_ylim(-0.5, len(tags) - 0.5)
ax.grid(True, alpha=0.3)

for _, row in plot_df_conversion.iterrows():
    if max_c == min_c:
        fontsize = 12
    else:
        fontsize = 9 + 9 * (row["incident_days"] - min_c) / (max_c - min_c)
    ax.text(
        week_to_x[row["year-week"]],
        tag_to_y[row["tags"]],
        str(int(row["incident_days"])),
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color="C0",
    )

ax.set_title("Conversion tag counts over time")
ax.set_xlabel("Year-Week")
ax.set_ylabel("Tag")
plt.tight_layout()
plt.show()

In [22]:
connections = sorted(plot_df_conversion["connection"].dropna().unique())
fig, axes = plt.subplots(len(connections), 1, figsize=(14, 4 * len(connections)), sharex=True)
if len(connections) == 1:
    axes = [axes]

for ax, conn in zip(axes, connections):
    sub = plot_df_conversion[plot_df_conversion["connection"] == conn]
    weeks = sorted(sub["year-week"].unique())
    tags = sorted(sub["tags"].unique())
    week_to_x = {w: i for i, w in enumerate(weeks)}
    tag_to_y = {t: i for i, t in enumerate(tags)}
    min_c, max_c = sub["incident_days"].min(), sub["incident_days"].max()

    ax.set_yticks(range(len(tags)))
    ax.set_yticklabels(tags)
    ax.set_xlim(-0.5, len(weeks) - 0.5)
    ax.set_ylim(-0.5, len(tags) - 0.5)
    ax.grid(True, alpha=0.3)
    ax.set_title(conn)

    for _, row in sub.iterrows():
        fontsize = 12 if max_c == min_c else 9 + 9 * (row["incident_days"] - min_c) / (max_c - min_c)
        ax.text(
            week_to_x[row["year-week"]],
            tag_to_y[row["tags"]],
            str(int(row["incident_days"])),
            ha="center", va="center", fontsize=fontsize, fontweight="bold", color="C0",
        )

axes[-1].set_xticks(range(len(weeks)))
axes[-1].set_xticklabels(weeks, rotation=90)
plt.tight_layout()
plt.show()

In [23]:
plot_df_causal = weekly_tags[
    (weekly_tags["tags"] != "no_pylon_ticket")
    & (weekly_tags["model_type"] == "causal")
].copy()
plot_df_causal["incident_days"] = plot_df_causal["incident_days"].astype(int)

weeks = sorted(plot_df_causal["year-week"].unique())
tags = sorted(plot_df_causal["tags"].unique())
week_to_x = {w: i for i, w in enumerate(weeks)}
tag_to_y = {t: i for i, t in enumerate(tags)}

min_c, max_c = plot_df_causal["incident_days"].min(), plot_df_causal["incident_days"].max()

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xticks(range(len(weeks)))
ax.set_xticklabels(weeks, rotation=90)
ax.set_yticks(range(len(tags)))
ax.set_yticklabels(tags)
ax.set_xlim(-0.5, len(weeks) - 0.5)
ax.set_ylim(-0.5, len(tags) - 0.5)
ax.grid(True, alpha=0.3)

for _, row in plot_df_causal.iterrows():
    if max_c == min_c:
        fontsize = 12
    else:
        fontsize = 9 + 9 * (row["incident_days"] - min_c) / (max_c - min_c)
    ax.text(
        week_to_x[row["year-week"]],
        tag_to_y[row["tags"]],
        str(int(row["incident_days"])),
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color="C0",
    )

ax.set_title("Conversion tag counts over time")
ax.set_xlabel("Year-Week")
ax.set_ylabel("Tag")
plt.tight_layout()
plt.show()

In [24]:
connections_causal = sorted(plot_df_causal["connection"].dropna().unique())
fig, axes = plt.subplots(len(connections_causal), 1, figsize=(14, 4 * len(connections_causal)), sharex=True)
if len(connections_causal) == 1:
    axes = [axes]

for ax, conn in zip(axes, connections_causal):
    sub = plot_df_causal[plot_df_causal["connection"] == conn]
    weeks = sorted(sub["year-week"].unique())
    tags = sorted(sub["tags"].unique())
    week_to_x = {w: i for i, w in enumerate(weeks)}
    tag_to_y = {t: i for i, t in enumerate(tags)}
    min_c, max_c = sub["incident_days"].min(), sub["incident_days"].max()

    ax.set_yticks(range(len(tags)))
    ax.set_yticklabels(tags)
    ax.set_xlim(-0.5, len(weeks) - 0.5)
    ax.set_ylim(-0.5, len(tags) - 0.5)
    ax.grid(True, alpha=0.3)
    ax.set_title(conn)

    for _, row in sub.iterrows():
        fontsize = 12 if max_c == min_c else 9 + 9 * (row["incident_days"] - min_c) / (max_c - min_c)
        ax.text(
            week_to_x[row["year-week"]],
            tag_to_y[row["tags"]],
            str(int(row["incident_days"])),
            ha="center", va="center", fontsize=fontsize, fontweight="bold", color="C0",
        )

axes[-1].set_xticks(range(len(weeks)))
axes[-1].set_xticklabels(weeks, rotation=90)
plt.tight_layout()
plt.show()

## Incidents by Month

In [25]:
monthly_incidents_workspaces = (
    dtw_daily_pylon.groupby(
        by=["account_name", "year", "month", "model_type", "connection"],
        as_index=False,
    )
    .agg(
        incident_days=("is_incident", "sum"),
        total_days=("is_incident", "count"),
    )
)
monthly_incidents_workspaces["incident_days"] = monthly_incidents_workspaces["incident_days"].astype(int)
monthly_incidents_workspaces["non_incident_days"] = (
    monthly_incidents_workspaces["total_days"] - monthly_incidents_workspaces["incident_days"]
)
monthly_incidents_workspaces["is_incident"] = (
    monthly_incidents_workspaces["incident_days"] / monthly_incidents_workspaces["total_days"]
)

monthly_incidents_workspaces["year-month"] = (
    monthly_incidents_workspaces["year"].astype(str)
    + "-"
    + monthly_incidents_workspaces["month"].astype(str).str.zfill(2)
)
monthly_incidents_workspaces["model_type-connection"] = (
    monthly_incidents_workspaces["model_type"].astype(str)
    + "-"
    + monthly_incidents_workspaces["connection"].astype(str).str.zfill(2)
)
monthly_incidents_workspaces = monthly_incidents_workspaces.sort_values(
    by=["year", "month"]
).reset_index(drop=True)

# create pivot table
monthly_incidents_workspaces_pivot = monthly_incidents_workspaces.pivot(index=["year-month","model_type","connection"], columns="account_name", values="is_incident")
monthly_incidents_workspaces_pivot=monthly_incidents_workspaces_pivot.reset_index().sort_values(by=["model_type","year-month"])
monthly_incidents_workspaces_pivot

In [26]:
monthly_incidents_workspaces.groupby(by=["year-month","model_type"])["is_incident"].describe().sort_values(by=["model_type","year-month"])

In [27]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
plt.suptitle("Overall Incident Rates per Month")
ax1.set_title("Incident Rate by Model Type")
sns.barplot(data=monthly_incidents_workspaces, x="year-month", y="is_incident", hue="model_type", ax=ax1, errorbar="sd")
plt.grid(True)
ax2.set_title("Incident Days by Model Type")
sns.barplot(data=monthly_incidents_workspaces, x="year-month", y="incident_days", hue="model_type", ax=ax2, errorbar="sd")
plt.grid(True)

In [28]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
monthly_incidents_workspaces = monthly_incidents_workspaces.sort_values(by=["year-month","connection"])
plt.suptitle("Overall Incident Rates per Month and Connection")
ax1.set_title("Incident Rate by Connection for conversion models")
sns.barplot(data=monthly_incidents_workspaces[monthly_incidents_workspaces["model_type"]=="conversion"], x="year-month", y="is_incident", hue="model_type-connection", ax=ax1, errorbar="sd")
plt.grid(True)
plt.legend(title="Connection", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
ax2.set_title("Incident Rate by Connection for causal models")
sns.barplot(data=monthly_incidents_workspaces[monthly_incidents_workspaces["model_type"]=="causal"], x="year-month", y="is_incident", hue="model_type-connection", ax=ax2, errorbar="sd")
plt.grid(True)
plt.legend(title="Connection", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()

## Incidents by Definition
- Combine Incidents as the following
- Get Start date and end date of incident. 
    - start_date: first incident after a false
    - end_date: if incident is false for three rows in a day
- Groupby account_name, audience_id
- Use dtw_daily_pylon and add according dtw_label and tags to the data

In [29]:
incidents_by_definition = combine_incidents(dtw_daily_pylon)
incidents_by_definition = incidents_by_definition.sort_values(by=["account_name", "audience_id", "start_date"])
incidents_by_definition["label"] = incidents_by_definition["model_type"] + " " + incidents_by_definition["period_type"]
incidents_by_definition


In [30]:
duration_stats = incidents_by_definition.groupby(by=["label"])["duration_days"].describe()
duration_stats

In [31]:
fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(1, 1, 1)
sns.histplot(data=incidents_by_definition, x="duration_days", hue="label", ax=ax, kde=True)
ax.set_title("Duration of Incidents by Model Type and Period Type")
ax.set_xlabel("Period Type")
ax.set_ylabel("Duration (days)")
plt.tight_layout()